# ATML PA3 - Task 3: Test-Time Compute: Sampling, Selection, and Adaptive Inference
## Kaggle Execution Notebook

This notebook runs the complete end-to-end pipeline for **Task 3** of EE-5102/CS-6304 (ATML PA3):
1. **Offline Test-Time Compute Scaling on MATH (1,000 problems x 32 rollouts)**:
   - Splits: 200 Dev / 800 Held-Out Eval / 50 Live Refinement
   - Evaluates budgets $N \in \{1, 2, 4, 8, 16, 32\}$ across 10 random permutations
   - Selection methods: FirstAns, Self-Consistency (majority vote), Highest-LogProb ($L_i$), Highest-SelfCertainty ($K_i$), SelfCertainty-RWC
   - Computes Oracle Coverage and decomposes errors into Sampling vs. Selection failures
2. **Offline Quality-Aware Selection & VisConf Scaling on MathVista (1,000 problems x 32 rollouts)**:
   - Per-token visual engagement $v_{i,t} = A^{\text{img}} / (A^{\text{img}} + A^{\text{prompt}} + A^{\text{gen}})$ and trajectory mean $V_i$
   - Candidate-pool calibrated self-certainty $c_{i,t} = \text{clip}_{[0,1]}\left(\frac{k_{i,t} - Q_{0.05}(G)}{\max(Q_{0.95}(G) - Q_{0.05}(G), 10^{-8})}\right)$
   - VisConf trajectory scoring $Z_i = V_i \times C_i$ and tokenwise fusion $Z^{\text{token}}_i = \frac{1}{T_i}\sum_t c_{i,t}v_{i,t}$
   - Selection methods: FirstAns, Self-Consistency, VisConf-Highest, VisConf-RWC, VisionOnly-RWC
3. **Signal Properties & Confidence-Vision Coupling (Step 3)**:
   - Association with length (shortest-rollout heuristic)
   - Top vs. bottom quartile accuracy differences
   - Mean tokenwise confidence-vision covariance $\text{Cov}_t(c_{i,t}, v_{i,t})$
4. **Live Generation & Sequential Refinement (50 MATH questions, Step 4)**:
   - Evaluates `Qwen/Qwen2.5-3B-Instruct` in FP16 under equalized token budget (~3,072 max new tokens):
     - **Strategy 1**: Single longer response (cap 3,072)
     - **Strategy 2**: Parallel sampling ($3 \times 1,024$, Self-Consistency)
     - **Strategy 3**: Sequential refinement (initial 1,024 + revision critique 2,048)
   - Analyzes wrong-to-correct and correct-to-wrong transitions
5. **Adaptive Inference Stopping Simulation (Step 5)**:
   - Stopping checked after $n \in \{4, 8, 16\}$ rollouts; otherwise stops at 32
   - Agreement controller (sweeps $\tau \in \{0.5, 0.625, 0.75, 0.875, 1.0\}$ on 200 Dev questions)
   - Quality-aware controller (sweeps $\tau$ and $\gamma \in \{0, 0.1, 0.25, 0.5\}$)
   - Held-out evaluation on 800 questions: accuracy, token cost, stopping distribution, and premature stopping rate
6. **Estimated Total Runtime**: $\approx 15\text{ to }20\text{ minutes}$ (Offline: ~3 min, Live: ~12-15 min on GPU).

### Step 0: Hardware Verification & Repository Setup

In [ ]:
# Verify GPU hardware
!nvidia-smi

import os
import sys

# Prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/ahmad-sarfraz345/ATML-PA3.git"
REPO_DIR = "/kaggle/working/ATML-PA3"

if os.path.exists("scripts/run_task3.py"):
    print("Already inside ATML-PA3 repository directory.")
    !git fetch origin && git reset --hard origin/main
elif os.path.exists(REPO_DIR):
    print("Updating existing repository in /kaggle/working/ATML-PA3...")
    %cd {REPO_DIR}
    !git fetch origin && git reset --hard origin/main
else:
    print("Cloning repository into /kaggle/working/ATML-PA3...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

### Step 1: Install Environment Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q pyarrow math-verify gdown
print("Task 3 dependencies successfully installed.")

### Step 2: Continuous Archiving Helper & Stage Controls

In [ ]:
import shutil
from pathlib import Path
import zipfile

def auto_package_task3_outputs(tag="latest"):
    """Packages lightweight results archive (< 10 MB) excluding heavy .pt checkpoints."""
    source_dir = Path("outputs/task3")
    if not source_dir.exists() or not any(source_dir.iterdir()):
        return
    zip_path = Path("/kaggle/working/task3_results.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in source_dir.rglob("*"):
            if f.is_file() and f.suffix not in [".pt", ".bin", ".pth"]:
                zf.write(f, f.relative_to(source_dir))
    size_mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"[Archive] Synced {zip_path} (tag: {tag}, {size_mb:.2f} MB)")

# Pipeline Stage Controls
STAGE_DOWNLOAD_DATA      = True
STAGE_OFFLINE_MATH       = True
STAGE_OFFLINE_MATHVISTA  = True
STAGE_ADAPTIVE_MATH      = True
STAGE_ADAPTIVE_MATHVISTA = True
STAGE_LIVE_REFINEMENT    = True
STAGE_ANALYSIS           = True

DATA_DIR = "data/task3"
OUTPUT_DIR = "outputs/task3"
SEED = 6304
LIVE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

### Step 3: Verify / Download Task 3 Parquet Dataset

Checks if the dataset is attached in `/kaggle/input/` or downloads directly from the official Google Drive folder via `gdown`.

In [ ]:
if STAGE_DOWNLOAD_DATA:
    print("--- Verifying Task 3 Dataset ---")
    !python -u -m scripts.run_task3 --stage download_data --data_dir {DATA_DIR} --seed {SEED}

### Step 4: Offline Test-Time Compute Scaling on MATH (Steps 1, 2, 3)

Evaluates $N \in \{1, 2, 4, 8, 16, 32\}$ across 10 random permutations for FirstAns, Self-Consistency, Highest-LogProb, Highest-SelfCertainty, and SelfCertainty-RWC. Analyzes Step 3 signal properties.

In [ ]:
if STAGE_OFFLINE_MATH:
    print("--- Executing Stage: Offline MATH Scaling ---")
    !python -u -m scripts.run_task3 --stage offline_math --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR} --seed {SEED}
    auto_package_task3_outputs("offline_math")

### Step 5: Offline Quality-Aware & VisConf Scaling on MathVista (Steps 1, 2, 3)

Evaluates $N \in \{1, 2, 4, 8, 16, 32\}$ across 10 random permutations for FirstAns, Self-Consistency, VisConf-Highest, VisConf-RWC, and VisionOnly-RWC. Analyzes tokenwise vs. trajectory VisConf and token covariance.

In [ ]:
if STAGE_OFFLINE_MATHVISTA:
    print("--- Executing Stage: Offline MathVista VisConf Scaling ---")
    !python -u -m scripts.run_task3 --stage offline_mathvista --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR} --seed {SEED}
    auto_package_task3_outputs("offline_mathvista")

### Step 6: Adaptive Inference Stopping Simulation (Step 5)

Sweeps stopping thresholds $\tau$ and $\gamma$ on the 200 Dev questions, freezes best controllers, and evaluates on 800 held-out questions for both MATH and MathVista.

In [ ]:
if STAGE_ADAPTIVE_MATH:
    print("--- Executing Stage: Adaptive Inference on MATH ---")
    !python -u -m scripts.run_task3 --stage adaptive_math --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR} --seed {SEED}
    auto_package_task3_outputs("adaptive_math")

if STAGE_ADAPTIVE_MATHVISTA:
    print("--- Executing Stage: Adaptive Inference on MathVista ---")
    !python -u -m scripts.run_task3 --stage adaptive_mathvista --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR} --seed {SEED}
    auto_package_task3_outputs("adaptive_mathvista")

### Step 7: Live Generation & Sequential Refinement on 50 MATH Problems (Step 4)

Loads `Qwen/Qwen2.5-3B-Instruct` in FP16 and runs live generation comparing:
1. Single longer response (cap 3,072 tokens)
2. Parallel sampling ($3 \times 1,024$ tokens, Self-Consistency)
3. Sequential refinement ($1,024$ tokens initial + $2,048$ tokens revision critique)
Takes $\approx 12\text{ to }15\text{ minutes}$ on a GPU.

In [ ]:
if STAGE_LIVE_REFINEMENT:
    print("--- Executing Stage: Live Inference & Sequential Refinement (50 Questions) ---")
    !python -u -m scripts.run_task3 --stage live_refinement --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR} --seed {SEED} --live_model_name {LIVE_MODEL}
    auto_package_task3_outputs("live_refinement")

### Step 8: Master Visualizations & Summary Synthesis

In [ ]:
if STAGE_ANALYSIS:
    print("--- Executing Stage: Synthesis & Visualizations ---")
    !python -u -m scripts.run_task3 --stage analysis --output_dir {OUTPUT_DIR}
    auto_package_task3_outputs("final")

# Display generated master summary and plots
from IPython.display import display, Markdown, Image

summary_file = Path(OUTPUT_DIR) / "task3_summary.md"
if summary_file.exists():
    display(Markdown(summary_file.read_text(encoding="utf-8")))

for img_name in ["scaling_curves.png", "failure_decomposition.png"]:
    img_path = Path(OUTPUT_DIR) / img_name
    if img_path.exists():
        print(f"\nFigure: {img_name}")
        display(Image(filename=str(img_path)))

### Step 9: Final Archive Verification

Verifies that `/kaggle/working/task3_results.zip` is created and $< 10\text{ MB}$.

In [ ]:
zip_path = Path("/kaggle/working/task3_results.zip")
if zip_path.exists():
    size_mb = zip_path.stat().st_size / (1024 * 1024)
    print(f"=== SUCCESS: Archive verified at {zip_path} ===")
    print(f"File size: {size_mb:.2f} MB (Strictly < 10 MB limit: {size_mb < 10.0})")
    print("\nArchive Contents:")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for info in zf.infolist():
            print(f"  - {info.filename} ({info.file_size / 1024:.1f} KB)")
else:
    print("ERROR: task3_results.zip was not found!")